In [1]:
import sys
import os
from typing import NamedTuple
import math
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import pickle

from sklearn.model_selection import TimeSeriesSplit, StratifiedKFold
import jax


sys.path.append(os.path.join(os.getcwd(), '../'))
from utils_worm import artificializer
from utils_worm import read_utils

sys.path.append(os.path.join(os.getcwd(), '../../../../'))
from utils import plot_utils
from utils import train_utils
from models import bom
from models import hgm


jax.config.update("jax_enable_x64", True)

In [2]:
import importlib
importlib.reload(artificializer)
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)
importlib.reload(bom)
importlib.reload(hgm)

<module 'models.hgm' from '/home/julian-zim/Files/Cloud/OneDrive/OneFiles/Linux/ZimmerLab/MSc_Thesis/Workspace/hobe/models/hgm.py'>

In [3]:
def seperate_predictions(file_ids, columns, models_spn_man_rec_res_pres):
	models_spn = [
		[
			pd.DataFrame(data=np.array(split[0])[None], columns=columns[i]) for split in models_spn_man_rec_res_pres[i]
		] for i in range(len(file_ids))
	]
	models_man = [
		[
			pd.DataFrame(data=np.array(split[1]), columns=[f'M{i}' for i in range(np.array(split[1]).shape[1])]) for split in models_spn_man_rec_res_pres[i]
		] for i in range(len(file_ids))
	]
	(models_rec,
	 models_res,
	 models_pres) = tuple(
		[
			[
				pd.DataFrame(data=np.array(split[m]), columns=columns[i]) for split in models_spn_man_rec_res_pres[i]
			] for i in range(len(file_ids))
		] for m in [2, 3, 4]
	)
	return models_spn, models_man, models_rec, models_res, models_pres

In [4]:
manual_hyperparameters = {
	'dimensions': 2,
	'loss_man_lambda': 0.0,
	'loss_res_lambda': 1.0 - 0.0,
	'wb_reg_lambda': 7.4E-3,
	'wm_reg_lambda': 1E-6,
}

ll = f'll{int(manual_hyperparameters['loss_man_lambda'] * 100):02d}'

with open('../sweep/hyperparams.pkl', 'rb') as f:
    swept_hyperparameters = pickle.load(f)

bom_reg_lambdas = {
	# 'wb_reg_lambda': manual_hyperparameters['wb_reg_lambda'],
	'wb_reg_lambda': 3.29E-2,
	# 'wb_reg_lambda': swept_hyperparameters[ll]['bom']['means']['wb_reg_lambda']['geometric'],
}
hgm_reg_lambdas = {
	'wb_reg_lambda': manual_hyperparameters['wb_reg_lambda'],
	# 'wb_reg_lambda': swept_hyperparameters[ll]['hgm']['means']['wb_reg_lambda']['geometric'],
	'wm_reg_lambda': manual_hyperparameters['wm_reg_lambda'],
	# 'wm_reg_lambda': swept_hyperparameters[ll]['hgm']['means']['wm_reg_lambda']['geometric'],
}

In [5]:
artificial = True
gcamp = True
gfp = True

organism = 'worm-2025-fieseler'

In [8]:
if gcamp:
	# setup
	file_ids_gcamp = list(range(0, 33+1))
	# file_ids_gcamp = [0,   # minrevs
	# 				  1,   # minswitch
	# 				  7,   # maxswitch
	# 				  11,  # median
	# 				  12,  # maxrevs
	# 				  17,  # mean
	# 				  19,  # paper
	# 				  21]  # clean

	path = os.path.join(read_utils.get_data_path(), 'exported_data_gcamp')

	bom_load_name_gcamp = os.path.join(organism, ll, f'ns_bom_gcamp_wb-{bom_reg_lambdas['wb_reg_lambda']:.2E}')
	hgm_load_name_gcamp = os.path.join(organism, ll, f'ns_hgm_gcamp_wb-{hgm_reg_lambdas['wb_reg_lambda']:.2E}_wm-{hgm_reg_lambdas['wm_reg_lambda']:.2E}')

	for file_id in file_ids_gcamp:
		# read
		(_, filename,
		 traces,
		 continuous_behaviors,
		 discrete_states, eigenworms,
		 kymographs) = [r[0] for r in read_utils.get_data(path, [file_id])]

		# clear nan values
		mask = eigenworms.notna().all(axis=1)
		traces = traces[mask]
		#continuous_behaviors = continuous_behaviors[mask]
		eigenworms = eigenworms[mask]
		discrete_states = discrete_states[mask]
		#kymographs = kymographs[mask]
		#filenames = filenames[mask]

		# naive ID unknown neurons
		traces.columns = pd.Index([
			f'{file_id:02d}_{n_id:03d}' if empty else label for n_id, (label, empty) in enumerate(zip(traces.columns, traces.columns == ''))
		])

		# read model
		boms_params = train_utils.load_params(bom_load_name_gcamp, [filename])[0]
		hgms_params = train_utils.load_params(hgm_load_name_gcamp, [filename])[0]

		# predict
		splits = list(StratifiedKFold(n_splits=5, shuffle=False).split(np.zeros(traces.shape[0]), discrete_states[['REV']].values))

		boms_spn_man_rec_res_pres = [
			bom.score_fn(params, traces.values[eval], eigenworms.values[eval]) for params, (train, eval) in zip(boms_params, splits)
		]

		hgms_spn_man_rec_res_pres = [
			hgm.score_fn(params, traces.values[eval], eigenworms.values[eval]) for params, (train, eval) in zip(hgms_params, splits)
		]

		(boms_spn_gcamp,
		 boms_man_gcamp,
		 boms_rec_gcamp,
		 boms_res_gcamp,
		 boms_pres_gcamp) = [r[0] for r in seperate_predictions([file_id], [traces.columns], [boms_spn_man_rec_res_pres])]

		(hgms_spn_gcamp,
		 hgms_man_gcamp,
		 hgms_rec_gcamp,
		 hgms_res_gcamp,
		 hgms_pres_gcamp) = [r[0] for r in seperate_predictions([file_id], [traces.columns], [hgms_spn_man_rec_res_pres])]

		train_utils.save_predictions(
			filename,
			(boms_spn_gcamp,
			 boms_man_gcamp,
			 boms_rec_gcamp,
			 boms_res_gcamp,
			 boms_pres_gcamp), 'bom', 'validation',
			save_path='../../out/predictions')

		train_utils.save_predictions(
			filename,
			(hgms_spn_gcamp,
			 hgms_man_gcamp,
			 hgms_rec_gcamp,
			 hgms_res_gcamp,
			 hgms_pres_gcamp), 'hgm', 'validation',
			save_path='../../out/predictions')

FileNotFoundError: [Errno 2] No such file or directory: '/home/julian-zim/Files/Cloud/OneDrive/OneFiles/Linux/ZimmerLab/MSc_Thesis/Workspace/hobe/utils/../organisms/worm-2025-fieseler/out/models/ll00/ns_bom_gcamp_wb-3.29E-02/params/2022-11-23_worm10.msgpack'

In [ ]:

if artificial:
	# setup
	dims_generation = 2
	weights = (0.1, -0.25)

	gen_ids_art = list(range(0, 9+1))

	# model setup
	bom_load_name_art = os.path.join(organism, ll, f'ns_bom_art_wb-{bom_reg_lambdas['wb_reg_lambda']:.2E}')
	hgm_load_name_art = os.path.join(organism, ll, f'ns_hgm_art_wb-{hgm_reg_lambdas['wb_reg_lambda']:.2E}_wm-{hgm_reg_lambdas['wm_reg_lambda']:.2E}')

	for file_id in gen_ids_art:

		# generate
		(filename_art,
		 neural_modes_art_,
		 traces_reconstructed_art_,
		 traces_residual_art_,
		 traces_art_,
		 params_art_,
		 params_cca_art_) = artificializer.artificialize(weights, hgm,
														 filenames_gcamp,
														 traces_gcamp[i].columns,
														 eigenworms_gcamp[i],
														 [ds[selection_bhl] for ds in discrete_states_gcamp][i],
														 n_dimensions=dims_generation)
		filenames_art.append(filename_art)

		modes_art.append(neural_modes_art_)
		traces_reconstructed_art.append(traces_reconstructed_art_)
		traces_residual_art.append(traces_residual_art_)
		traces_art.append(traces_art_)

		file_ids_art.append(file_ids_gcamp[i])

		discrete_states_art.append(discrete_states_gcamp[i].copy())
		eigenworms_art.append(eigenworms_gcamp[i].copy())







	# read model
	boms_params_art = train_utils.load_params(bom_load_name_art, filenames_art)

	boms_losses_art = train_utils.load_losses(bom_load_name_art, filenames_art)
	boms_losses_train_art = [losses[0] for losses in boms_losses_art]
	boms_man_losses_train_art = [losses[1] for losses in boms_losses_art]
	boms_res_losses_train_art = [losses[2] for losses in boms_losses_art]
	boms_mpe_losses_train_art = [losses[3] for losses in boms_losses_art]
	boms_losses_test_art = [losses[4] for losses in boms_losses_art]
	boms_man_losses_test_art = [losses[5] for losses in boms_losses_art]
	boms_res_losses_test_art = [losses[6] for losses in boms_losses_art]
	boms_mpe_losses_test_art = [losses[7] for losses in boms_losses_art]

	hgms_params_art = train_utils.load_params(hgm_load_name_art, filenames_art)

	hgms_losses_art = train_utils.load_losses(hgm_load_name_art, filenames_art)
	hgms_losses_train_art = [losses[0] for losses in hgms_losses_art]
	hgms_man_losses_train_art = [losses[1] for losses in hgms_losses_art]
	hgms_res_losses_train_art = [losses[2] for losses in hgms_losses_art]
	hgms_mpe_losses_train_art = [losses[3] for losses in hgms_losses_art]
	hgms_losses_test_art = [losses[4] for losses in hgms_losses_art]
	hgms_man_losses_test_art = [losses[5] for losses in hgms_losses_art]
	hgms_res_losses_test_art = [losses[6] for losses in hgms_losses_art]
	hgms_mpe_losses_test_art = [losses[7] for losses in hgms_losses_art]


In [12]:
gfp = True
if gfp:
	read_ids_gfp = list(range(0, 7+1))
	# read_ids_gfp = [2, 4, 6]
	# read_ids_gfp = [2]
	plot_id_gfp = 2

In [13]:
# read
if gfp:
	path = os.path.join(read_utils.get_data_path(), 'exported_data_gfp')
	file_ids_gfp, filenames_gfp = read_utils.get_info(path, read_ids_gfp)


In [7]:
organism = 'worm-2025-fieseler'

if gfp:
	bom_load_name_gfp = os.path.join(organism, ll, f'ns_bom_gfp_wb-{bom_reg_lambdas['wb_reg_lambda']:.2E}')
	hgm_load_name_gfp = os.path.join(organism, ll, f'ns_hgm_gfp_wb-{hgm_reg_lambdas['wb_reg_lambda']:.2E}_wm-{hgm_reg_lambdas['wm_reg_lambda']:.2E}')

In [ ]:
if gfp:
	boms_params_gfp = train_utils.load_params(bom_load_name_gfp, filenames_gfp)

	boms_losses_gfp = train_utils.load_losses(bom_load_name_gfp, filenames_gfp)
	boms_losses_train_gfp = [losses[0] for losses in boms_losses_gfp]
	boms_man_losses_train_gfp = [losses[1] for losses in boms_losses_gfp]
	boms_res_losses_train_gfp = [losses[2] for losses in boms_losses_gfp]
	boms_mpe_losses_train_gfp = [losses[3] for losses in boms_losses_gfp]
	boms_losses_test_gfp = [losses[4] for losses in boms_losses_gfp]
	boms_man_losses_test_gfp = [losses[5] for losses in boms_losses_gfp]
	boms_res_losses_test_gfp = [losses[6] for losses in boms_losses_gfp]
	boms_mpe_losses_test_gfp = [losses[7] for losses in boms_losses_gfp]

	hgms_params_gfp = train_utils.load_params(hgm_load_name_gfp, filenames_gfp)

	hgms_losses_gfp = train_utils.load_losses(hgm_load_name_gfp, filenames_gfp)
	hgms_losses_train_gfp = [losses[0] for losses in hgms_losses_gfp]
	hgms_man_losses_train_gfp = [losses[1] for losses in hgms_losses_gfp]
	hgms_res_losses_train_gfp = [losses[2] for losses in hgms_losses_gfp]
	hgms_mpe_losses_train_gfp = [losses[3] for losses in hgms_losses_gfp]
	hgms_losses_test_gfp = [losses[4] for losses in hgms_losses_gfp]
	hgms_man_losses_test_gfp = [losses[5] for losses in hgms_losses_gfp]
	hgms_res_losses_test_gfp = [losses[6] for losses in hgms_losses_gfp]
	hgms_mpe_losses_test_gfp = [losses[7] for losses in hgms_losses_gfp]